In [39]:
import numpy as np
import random

In [40]:
class CliqueGenerator():
    """
    Create an adjacency matrix with cliques in it to test our algorithm. It will have 'num_cliques' 
    cliques with random size between 3 and max_clique_size.
    This function returns the adjacency matrix, the solutions and the sizes of the cliques.
    """
    def __init__(self, 
                 noise_level, 
                 adjacency_matrix_size, 
                 num_cliques,
                 max_clique_size, 
                 info=False,
                diag_cut=False):
        
        self.noise_level = noise_level
        self.adjacency_matrix_size = adjacency_matrix_size
        self.num_cliques = num_cliques
        self.info = info
        self.max_clique_size = max_clique_size
        
    def create(self,num_cliques=None,diag_cut=True):
        if num_cliques is None:
            num_cliques = self.num_cliques
        assert num_cliques <= self.adjacency_matrix_size
        placeholder_array = np.zeros((self.adjacency_matrix_size,self.adjacency_matrix_size))
        clique_sizes = []
        for i in range(num_cliques):
            clique_sizes.append(int(np.random.randint(3,self.max_clique_size)))
        
        candidates = []
        i = 0
        while len(candidates)<num_cliques:
            candidate = []
            while len(candidate) < clique_sizes[i]:
                tmp = int(np.random.randint(0,self.adjacency_matrix_size))
                if tmp not in candidate:
                    candidate.append(tmp)
                candidate.sort()
            if candidate not in candidates:
                candidates.append(candidate)
                i+=1
        
        
        noise = int((1-self.noise_level)*100)
        noise_matrix = np.random.randint(0,100,size=(self.adjacency_matrix_size,self.adjacency_matrix_size))>noise
        difference = noise_matrix-placeholder_array*noise_matrix
        placeholder_array += difference
        for candidate in candidates:
            for i in candidate:
                for j in candidate:
                    if j!=i:
                        placeholder_array[i][j]=1
        if self.info:
            print("[",len(clique_sizes),"]","Cliques will be created in the following sizes: ",*clique_sizes,"\n\n","-"*30)
            print("The solutions will be: ", candidates,"\n\n","-"*30,"\n")  
            print("Now we have a pure adjacency matrix without any noise:","\n\n","-"*30,"\n",placeholder_array,"\n\n","-"*30)


        
        
        if self.info:
            print("\n\n","-"*30,"\nNow we have a pure adjacency matrix with noise:","\n\n","-"*30,"\n",placeholder_array,"\n\n","-"*30)
        if diag_cut:
            for i in range(placeholder_array.shape[0]):
                placeholder_array[i][i]=0#edges from i to i don't contribute to the solution, so we will ignore them
        
        return placeholder_array, candidates, clique_sizes

In [41]:
def reorder_adjacency_matrix(adjacency_matrix,diag_cut=True):
    original = adjacency_matrix
    if diag_cut:
        for i in range(adjacency_matrix.shape[0]):
            original[i][i]=0#edges from i to i don't contribute to the solution, so we will ignore them
    #we want to sort the matrix by the amount of connections the rows have. ofc this has to happen
    #in the same way with the columns.
    original_order = np.arange(adjacency_matrix.shape[0],dtype=np.int32)#this is actually redundant because we have the index.
    post_order = np.argsort(adjacency_matrix.sum(axis=1,dtype=np.int64))[::-1]#this is the index-set we will use
    amount_post_order = np.sort(adjacency_matrix.sum(axis=1,dtype=np.int32))[::-1]#these are the numbers we will use to replace the 1's

    
    sorted_matrix = original[post_order,:][:,post_order].copy()#now the row with the most connections is at the top
    #while rows with less connections follow in descending order
    #the cols are sorted too, so we don't have to worry about translation errors.
    
    return sorted_matrix, original_order, post_order, amount_post_order# we'll need all of them

In [42]:
def select_and_reassign_significant_part(sorted_adjacency_matrix,amount_post_order):
    sorted_adjacency_matrix = sorted_adjacency_matrix.copy()
    amount_post_order = amount_post_order.copy()
    for i, row in enumerate(sorted_adjacency_matrix):
        sorted_adjacency_matrix[i]=row*amount_post_order
    last_non_trivial_row = (amount_post_order>=2).sum()
    idx = np.arange(last_non_trivial_row,dtype=np.int32)
    sorted_adjacency_matrix = sorted_adjacency_matrix[idx,:][:,idx]
    return sorted_adjacency_matrix, amount_post_order[:last_non_trivial_row]

In [43]:
def compute_incoming_connections(adjacency_matrix):
    incoming_connections = (adjacency_matrix!=0).sum(axis=0)
    sorted_numerated_in_matrix = (adjacency_matrix!=0)*incoming_connections
    return sorted_numerated_in_matrix,incoming_connections

In [431]:
class Solver():
    def __init__(self,
                 noise_level = 0.05, 
                 adjacency_matrix_size = 40, 
                 num_cliques = 5, 
                 max_clique_size = 5,
                 clique=None,
                 info=False,
                diag_cut=True):
        
        self.diag_cut = diag_cut
        if clique is None:
            self.gen = CliqueGenerator(noise_level = noise_level, 
                                   adjacency_matrix_size = adjacency_matrix_size, 
                                   num_cliques = num_cliques, 
                                   max_clique_size = max_clique_size,
                                   info=info)
        
        
            self.adjacency_matrix, self.solutions, self.clique_sizes = self.gen.create()
        else:
            self.adjacency_matrix = clique
        
        #from here on, only adjacency matrix is used. the code above is just to give you an example
        
        self.sorted_adjacency_matrix, self.original_order, self.post_order, self.amount_post_order = reorder_adjacency_matrix(self.adjacency_matrix,diag_cut=diag_cut)
        self.sorted_numerated_out_matrix, self.outgoing_connections = select_and_reassign_significant_part(self.sorted_adjacency_matrix,self.amount_post_order)
        self.sorted_numerated_in_matrix, self.incoming_connections = compute_incoming_connections(self.sorted_numerated_out_matrix)
        self.valid_index = np.arange(self.sorted_numerated_out_matrix.shape[0])
        
        self.nodes_of_interest = (self.outgoing_connections>1)&(self.incoming_connections>1)
        self.sorted_numerated_out_matrix=self.sorted_numerated_out_matrix[self.nodes_of_interest,:][:,self.nodes_of_interest]
        self.sorted_numerated_in_matrix=self.sorted_numerated_in_matrix[self.nodes_of_interest,:][:,self.nodes_of_interest]
        self.valid_index=self.valid_index[self.nodes_of_interest]
        self.exclusion_prep = self.post_order[:len(self.outgoing_connections)]
        
        self.not_included = list(self.exclusion_prep[~self.nodes_of_interest])
        self.exclusion_prep = self.exclusion_prep[self.nodes_of_interest]
        self.outgoing_connections = self.outgoing_connections[self.nodes_of_interest]
        self.incoming_connections = self.incoming_connections[self.nodes_of_interest]
        self.nodes_of_interest = self.nodes_of_interest[self.nodes_of_interest]
        self.valid_index=self.valid_index[self.nodes_of_interest]
        #self.info_print()
        self.helper_adjacency_matrix = None
        
        self.cliques = []
        
        
    def info_print(self):
        print("-"*60)
        self.out_matrix()
        print("-"*60)
        self.in_matrix()
        print("-"*60)
        self.interest()
        print("-"*60)
        print(self.exclusion_prep, self.not_included,self.nodes_of_interest)
        
    def out_matrix(self):
        print(self.sorted_numerated_out_matrix,"\n",self.sorted_numerated_out_matrix.shape,"\n")
    def in_matrix(self):
        print(self.sorted_numerated_in_matrix,"\n",self.sorted_numerated_in_matrix.shape,"\n")
    def interest(self):
        print(self.nodes_of_interest)

    def correct_out(self,col):
        mask = self.sorted_numerated_out_matrix[:,col]!=0
        self.sorted_numerated_out_matrix[:,col]-=mask
        self.outgoing_connections[col]-=1
        
    def correct_in(self,col):
        mask = self.sorted_numerated_in_matrix[:,col]!=0
        self.sorted_numerated_in_matrix[:,col]-=mask
        self.incoming_connections[col]-=1
        
    def remove_idx(self,idx):
        assert self.sorted_numerated_out_matrix.shape[0]>0
        idx_len = self.sorted_numerated_out_matrix.shape[0]
        valid_idx = np.ones(idx_len,dtype=np.int32)
        valid_idx[idx]=0
        valid_idx = np.bool(valid_idx)
        for i,b in enumerate(self.sorted_numerated_out_matrix[idx]!=0):
            if b:
                self.correct_in(i)
        self.sorted_numerated_out_matrix = self.sorted_numerated_out_matrix[valid_idx,:][:,valid_idx]
        self.sorted_numerated_in_matrix = self.sorted_numerated_in_matrix[valid_idx,:][:,valid_idx]
        
                
        self.not_included.append(list(self.exclusion_prep[~valid_idx]))
        self.exclusion_prep = self.exclusion_prep[valid_idx]
        self.outgoing_connections = self.outgoing_connections[valid_idx]
        self.incoming_connections = self.incoming_connections[valid_idx]
        self.nodes_of_interest = self.nodes_of_interest[valid_idx]
        
    def update_nodes_of_interest(self):
        self.nodes_of_interest = (self.outgoing_connections>1)&(self.incoming_connections>1)
        mask = ~self.nodes_of_interest
        for i,b in enumerate(mask):
            if b:
                self.remove_idx(i)
    
    def create_helper(self):
        self.helper_adjacency_matrix = self.sorted_numerated_in_matrix != 0
        
    def check_idx(self,idx):
        assert idx<len(self.outgoing_connections)
        print("1) idx",idx)
        print("#"*40)
        #to avoid repetetive row computations
        if self.helper_adjacency_matrix is None:
            self.create_helper()
        if len(self.outgoing_connections)!=len(self.helper_adjacency_matrix[0]):
            self.create_helper()
        print("2) helper_adjacency_matrix\n",self.helper_adjacency_matrix,"\n len(self.outgoing_connections): ",len(self.outgoing_connections))
    
        print("#"*40)
        lst = self.helper_adjacency_matrix[idx]#get the specific row we want to work on
        print("lst: self.helper_adjacency_matrix[idx]",lst)
        sum_out = lst.sum()#amount of outgoing connections
        print("sum_out: lst.sum(): ",sum_out)
        check = 0#here : counter for suitable candidates
        indices = []#for the outer matrix
        candidates = []#if a node is suitable, it will appear here
        #print(lst)
        print("-"*40)

        print("check = 0, indices = [], candidates = []\n\nbegin for i,b in enumerate(lst)")
        for i,b in enumerate(lst):
            if b:
                if sum_out<2:
                    print("if b: if sum_out < 2: break")
                    break
                if self.helper_adjacency_matrix[i][idx]:
                    print("ansonsten : wenn self.sorted_numerated_out_matrix[i][idx] ungleich 0: dann ##",self.sorted_numerated_out_matrix[i][idx])
                    indices.append(i)
                    check+=1
                    print("indices append, check+=1")
                else:
                    sum_out-=1
                    print("else sum_out-=1")
        if check<2:
            print("Wenn Check<2 dann self.remove_idx(idx), sollte hier das ende sein.")
            self.remove_idx(idx)#das richtig
            return
        else:
            print("%"*40)

            print("Check offensichtlich größer als 2")
            print("begin der reversed(indices) schleife. für index in ",reversed(indices))
            for index in reversed(indices):#begin with the smallest. the indices with the most connections are to the right
                tmp_bool_vec = self.helper_adjacency_matrix[idx]&self.helper_adjacency_matrix[index]
                tmp_bool_sum = tmp_bool_vec.sum()
                print("tmp_bool_vec ",tmp_bool_vec)
                print("tmp_bool_sum ",tmp_bool_sum)
                if tmp_bool_sum>1:
                    print("$"*40)
                    print("Da tmp_bool_sum > 1: ")
                    tmp_bool_vec2 = tmp_bool_vec
                    tmp_bool_vec2[idx]=1
                    tmp_bool_vec2[index]=1
                    print(f"tmp_bool_vec2 wurde befüllt an {idx} und {index}: ",tmp_bool_vec2)
                    count=len(self.helper_adjacency_matrix[0])-1
                    print("count=len(self.helper_adjacency_matrix[0])-1",count)
                    print("."*40)
                    while tmp_bool_sum and count>=0:
                        print("while tmp_bool_sum and count>=0:")
                        if tmp_bool_vec[count]==1 and count!=index and count!=idx:
                            tmp_bool_sum-=1
                            print("if tmp_bool_vec[count]==1 and count!=index and count!=idx: tmp_bool_sum-=1: ",tmp_bool_sum)
                            if (self.helper_adjacency_matrix[count]&tmp_bool_vec2).sum()>=2:
                                print("if (self.helper_adjacency_matrix[count]&tmp_bool_vec2).sum()>=2:")
                                if count not in candidates and count!=idx and count!=index:
                                    print("if count not in candidates and count!=idx and count!=index: candidates.append(count)")
                                    candidates.append(count)
                                    if index not in candidates:
                                        candidates.append(index)#does basically the same, just marking it
                                else:
                                    print("ELSE: self.correct_in(idx)")
                                    self.correct_in(idx)
                        
                        count-=1
                        print("unterstes ende while: count", count)
                        print("."*40)
                    check-=1
                    print("+"*40)

                    if check<2 and len(candidates)<1:#clique begins with 3
                        self.remove_idx(idx)
                        print("Wenn check<2 und len(candidates)<1, dann remove_idx(idx) - normal, hier gibts nichts zu sehen")
                        return
                    print("+"*40)

                #print(candidates)
                #print("idx: ",idx)
                #print("index",index)
                for c in candidates:
                    if c in self.not_included:
                        candidates.remove(c)
        print("?"*40)
        
        candidates.sort()# because again, low indeces mean less connections, this is where we wlll start to eliminate
        print("Wir sind außerhalb der while. sortieren jetzt candidates um von rechts nach links zu arbeiten ",candidates,"\n\n")
        if idx not in candidates:
            candidates.append(idx)#we want it to be the most right one
        print("Wir appenden idx an candidates: ",candidates)
        checkbox = np.zeros((len(candidates),len(candidates)))
        
        for i in range(len(candidates)):
            for j in range(len(candidates)):
                checkbox[i][j]=self.sorted_numerated_out_matrix[candidates[i]][candidates[j]]!=0
                
            
        checkbox[len(candidates)-1][len(candidates)-1]=1
        checkbox = np.array(checkbox,dtype=np.int32)
    
        print("jetzt haben wir ne Checkbox mit den richtigen EInträgen: \n",checkbox)
        
        real_candidates = np.array(checkbox.sum(axis=0)>1,dtype=np.int32)
        print("\n Und wir haben real_candidates = np.array(checkbox.sum(axis=0)>1,dtype=np.int32): ",real_candidates)
        not_suitable_candidates = []
        #for c in candidates[real_candidates]:
            #not_suitable_candidates.append(c)
        #candidates = candidates[real_candidates]
        #print(real_candidates)

        print("?"*40)
        print("&"*40)

        length = len(real_candidates)
        print("Die Länge davon: length = ",length)
        for i,candidate in enumerate(reversed(candidates)):
            c=length-i-1
            if real_candidates[c]:
                
                print(checkbox[:,c])
                if c == length-1:
                    continue
                print(len(checkbox[:,c]),checkbox[:,c])
                print(len(checkbox[:,length-1]),checkbox[:,length-1])
                check = checkbox[:,c]&checkbox[:,length-1]
                check_sum = (check).sum()
                print(check, check_sum)
                if check_sum < 1:
                    print("idx: ",candidates[i])
                    
        # DAMN IT, retry
        print("&"*40)
        print("Insgesamt wäre das mapping: ")
        for i in candidates:
            print(i," : ",self.outgoing_connections[i])

        #self.remove_idx(idx)
        self.update_nodes_of_interest()#at the end     
        
        
        
        
        
        
        
        
        
    def pick_last(self):
        last_candidate = len(self.outgoing_connections)-1
        self.check_idx(last_candidate)

In [432]:
S = Solver()
S.check_idx(4)
print(S.solutions)

1) idx 4
########################################
2) helper_adjacency_matrix
 [[False False False  True  True  True False False False False  True  True
   True False False False False False False False  True]
 [False False  True False False False False  True False False False False
  False  True  True False False  True False False False]
 [False  True False False False False False  True  True False False False
  False False  True False False False False False False]
 [ True False False False False False False False False False False  True
  False False False False False False  True False False]
 [False False False False False False  True False  True  True False False
  False False  True False False False False False False]
 [ True  True False False False False False  True False False  True False
   True False False False False False False False False]
 [False False False False  True False False False  True  True False False
  False False False False False False False False False]
 [Fal

In [ ]:
    def check_idx(self,idx):
        assert idx<len(self.outgoing_connections)
        if self.helper_adjacency_matrix is None:
            self.create_helper()
        if len(self.outgoing_connections)!=len(self.helper_adjacency_matrix[0]):
            self.create_helper()
        lst = self.helper_adjacency_matrix[idx]
        sum_out = lst.sum()
        check = 0
        indices = []
        candidates = []
        print(lst)
        for i,b in enumerate(lst):
            if b:
                if sum_out<2:
                    break
                if self.helper_adjacency_matrix[i][idx]:
                    print("##",self.sorted_numerated_out_matrix[i][idx])
                    indices.append(i)
                    check+=1
                else:
                    sum_out-=1
        if check<2:
            self.remove_idx(idx)
            return
        else:
            for index in reversed(indices):#begin with the smallest. the indices with the most connections are to the right
                tmp_bool_vec = self.helper_adjacency_matrix[idx]&self.helper_adjacency_matrix[index]
                tmp_bool_sum = tmp_bool_vec.sum()
                if tmp_bool_sum>1:
                    
                    tmp_bool_vec2 = tmp_bool_vec
                    tmp_bool_vec2[idx]=1
                    tmp_bool_vec2[index]=1
                    count=len(self.helper_adjacency_matrix[0])-1
                    while tmp_bool_sum and count>=0:
                        if tmp_bool_vec[count]==1 and count!=index and count!=idx:
                            tmp_bool_sum-=1
                            if (self.helper_adjacency_matrix[count]&tmp_bool_vec2).sum()>=2:
                                if count not in candidates and count!=idx and count!=index:
                                    candidates.append(count)
                                    if index not in candidates:
                                        candidates.append(np.array([index]))#does basically the same, just marking it
                                else:
                                    self.correct_in(idx)
                        count-=1
                    
                    check-=1
                    if check<2 and len(candidates)<1:#clique begins with 3
                        self.remove_idx(idx)
                        return
                #print(candidates)
                #print("idx: ",idx)
                #print("index",index)
                for c in candidates:
                    if c in self.not_included:
                        candidates.remove(c)
                
        candidates.sort()# because again, low indeces mean less connections, this is where we wlll start to eliminate
        
        if idx not in candidates:
            candidates.append(idx)#we want it to be the most right one
        checkbox = np.zeros((len(candidates),len(candidates)))
        print(candidates)
        for i in range(len(candidates)):
            for j in range(len(candidates)):
                checkbox[i][j]=self.sorted_numerated_out_matrix[i][j]!=0
        checkbox = np.array(checkbox,dtype=np.int32)
        print(checkbox)
        real_candidates = np.array(checkbox.sum(axis=0)>1,dtype=np.int32)
        
        not_suitable_candidates = []
        for c in candidates[real_candidates]:
            not_suitable_candidates.append(c)
        candidates = candidates[real_candidates]
        print(real_candidates)

            
        length = len(real_candidates)
        print(length)
        for i,candidate in enumerate(reversed(range(length))):
            if real_candidates[candidate]:
                c=length-i-1
                print(checkbox[:,c])
                if c == length-1:
                    continue
                print(len(checkbox[:,c]),checkbox[:,c])
                print(len(checkbox[:,length-1]),checkbox[:,length-1])
                check = checkbox[:,c]&checkbox[:,length-1]
                check_sum = (check).sum()
                print(check, check_sum)
                if check_sum < 2:
                    print("idx: ",)
                    
        
        #self.remove_idx(idx)
        self.update_nodes_of_interest()#at the end     
        

In [ ]:
class Solver():
    def __init__(self,
                 noise_level = 0.1, 
                 adjacency_matrix_size = 20, 
                 num_cliques = 5, 
                 max_clique_size = 5,
                 clique=None,
                 info=False,
                diag_cut=True):
        
        self.diag_cut = diag_cut
        if clique is None:
            self.gen = CliqueGenerator(noise_level = noise_level, 
                                   adjacency_matrix_size = adjacency_matrix_size, 
                                   num_cliques = num_cliques, 
                                   max_clique_size = max_clique_size,
                                   info=info)
        
        
            self.adjacency_matrix, self.solutions, self.clique_sizes = self.gen.create()
        else:
            self.adjacency_matrix = clique
        
        #from here on, only adjacency matrix is used. the code above is just to give you an example
        
        self.sorted_adjacency_matrix, self.original_order, self.post_order, self.amount_post_order = reorder_adjacency_matrix(self.adjacency_matrix,diag_cut=diag_cut)
        self.sorted_numerated_out_matrix, self.outgoing_connections = select_and_reassign_significant_part(self.sorted_adjacency_matrix,self.amount_post_order)
        self.sorted_numerated_in_matrix, self.incoming_connections = compute_incoming_connections(self.sorted_numerated_out_matrix)
        self.valid_index = np.arange(self.sorted_numerated_out_matrix.shape[0])
        
        self.nodes_of_interest = (self.outgoing_connections>1)&(self.incoming_connections>1)
        self.sorted_numerated_out_matrix=self.sorted_numerated_out_matrix[self.nodes_of_interest,:][:,self.nodes_of_interest]
        self.sorted_numerated_in_matrix=self.sorted_numerated_in_matrix[self.nodes_of_interest,:][:,self.nodes_of_interest]
        self.valid_index=self.valid_index[self.nodes_of_interest]
        self.exclusion_prep = self.post_order[:len(self.outgoing_connections)]
        
        self.not_included = list(self.exclusion_prep[~self.nodes_of_interest])
        self.exclusion_prep = self.exclusion_prep[self.nodes_of_interest]
        self.outgoing_connections = self.outgoing_connections[self.nodes_of_interest]
        self.incoming_connections = self.incoming_connections[self.nodes_of_interest]
        self.nodes_of_interest = self.nodes_of_interest[self.nodes_of_interest]
        self.valid_index=self.valid_index[self.nodes_of_interest]
        #self.info_print()
        self.helper_adjacency_matrix = None
        
        self.cliques = []
        
        
    def info_print(self):
        print("-"*60)
        self.out_matrix()
        print("-"*60)
        self.in_matrix()
        print("-"*60)
        self.interest()
        print("-"*60)
        print(self.exclusion_prep, self.not_included,self.nodes_of_interest)
        
    def out_matrix(self):
        print(self.sorted_numerated_out_matrix,"\n",self.sorted_numerated_out_matrix.shape,"\n")
    def in_matrix(self):
        print(self.sorted_numerated_in_matrix,"\n",self.sorted_numerated_in_matrix.shape,"\n")
    def interest(self):
        print(self.nodes_of_interest)

    def correct_out(self,col):
        mask = self.sorted_numerated_out_matrix[:,col]!=0
        self.sorted_numerated_out_matrix[:,col]-=mask
        self.outgoing_connections[col]-=1
        
    def correct_in(self,col):
        mask = self.sorted_numerated_in_matrix[:,col]!=0
        self.sorted_numerated_in_matrix[:,col]-=mask
        self.incoming_connections[col]-=1
        
    def remove_idx(self,idx):
        assert self.sorted_numerated_out_matrix.shape[0]>0
        idx_len = self.sorted_numerated_out_matrix.shape[0]
        valid_idx = np.ones(idx_len,dtype=np.int32)
        valid_idx[idx]=0
        valid_idx = np.bool(valid_idx)
        for i,b in enumerate(self.sorted_numerated_out_matrix[idx]!=0):
            if b:
                self.correct_in(i)
        self.sorted_numerated_out_matrix = self.sorted_numerated_out_matrix[valid_idx,:][:,valid_idx]
        self.sorted_numerated_in_matrix = self.sorted_numerated_in_matrix[valid_idx,:][:,valid_idx]
        
                
        self.not_included.append(list(self.exclusion_prep[~valid_idx]))
        self.exclusion_prep = self.exclusion_prep[valid_idx]
        self.outgoing_connections = self.outgoing_connections[valid_idx]
        self.incoming_connections = self.incoming_connections[valid_idx]
        self.nodes_of_interest = self.nodes_of_interest[valid_idx]
        
    def update_nodes_of_interest(self):
        self.nodes_of_interest = (self.outgoing_connections>1)&(self.incoming_connections>1)
        mask = ~self.nodes_of_interest
        for i,b in enumerate(mask):
            if b:
                self.remove_idx(i)
    
    def create_helper(self):
        self.helper_adjacency_matrix = self.sorted_numerated_in_matrix != 0
        
    def check_idx(self,idx):
        assert idx<len(self.outgoing_connections)
        print("1) idx",idx)
        print("#"*40)
        #to avoid repetetive row computations
        if self.helper_adjacency_matrix is None:
            self.create_helper()
        if len(self.outgoing_connections)!=len(self.helper_adjacency_matrix[0]):
            self.create_helper()
        print("2) helper_adjacency_matrix\n",self.helper_adjacency_matrix,"\n len(self.outgoing_connections): ",len(self.outgoing_connections))
    
        print("#"*40)
        lst = self.helper_adjacency_matrix[idx]#get the specific row we want to work on
        print("lst: self.helper_adjacency_matrix[idx]",lst)
        sum_out = lst.sum()#amount of outgoing connections
        print("sum_out: lst.sum(): ",sum_out)
        check = 0#here : counter for suitable candidates
        indices = []#for the outer matrix
        candidates = []#if a node is suitable, it will appear here
        #print(lst)
        print("-"*40)

        print("check = 0, indices = [], candidates = []\n\nbegin for i,b in enumerate(lst)")
        for i,b in enumerate(lst):
            if b:
                if sum_out<2:
                    print("if b: if sum_out < 2: break")
                    break
                if self.helper_adjacency_matrix[i][idx]:
                    print("ansonsten : wenn self.sorted_numerated_out_matrix[i][idx] ungleich 0: dann ##",self.sorted_numerated_out_matrix[i][idx])
                    indices.append(i)
                    check+=1
                    print("indices append, check+=1")
                else:
                    sum_out-=1
                    print("else sum_out-=1")
        if check<2:
            print("Wenn Check<2 dann self.remove_idx(idx), sollte hier das ende sein.")
            self.remove_idx(idx)#das richtig
            return
        else:
            print("%"*40)

            print("Check offensichtlich größer als 2")
            print("begin der reversed(indices) schleife. für index in ",reversed(indices))
            for index in reversed(indices):#begin with the smallest. the indices with the most connections are to the right
                tmp_bool_vec = self.helper_adjacency_matrix[idx]&self.helper_adjacency_matrix[index]
                tmp_bool_sum = tmp_bool_vec.sum()
                print("tmp_bool_vec ",tmp_bool_vec)
                print("tmp_bool_sum ",tmp_bool_sum)
                if tmp_bool_sum>1:
                    print("$"*40)
                    print("Da tmp_bool_sum > 1: ")
                    tmp_bool_vec2 = tmp_bool_vec
                    tmp_bool_vec2[idx]=1
                    tmp_bool_vec2[index]=1
                    print(f"tmp_bool_vec2 wurde befüllt an {idx} und {index}: ",tmp_bool_vec2)
                    count=len(self.helper_adjacency_matrix[0])-1
                    print("count=len(self.helper_adjacency_matrix[0])-1",count)
                    print("."*40)
                    while tmp_bool_sum and count>=0:
                        print("while tmp_bool_sum and count>=0:")
                        if tmp_bool_vec[count]==1 and count!=index and count!=idx:
                            tmp_bool_sum-=1
                            print("if tmp_bool_vec[count]==1 and count!=index and count!=idx: tmp_bool_sum-=1: ",tmp_bool_sum)
                            if (self.helper_adjacency_matrix[count]&tmp_bool_vec2).sum()>=2:
                                print("if (self.helper_adjacency_matrix[count]&tmp_bool_vec2).sum()>=2:")
                                if count not in candidates and count!=idx and count!=index:
                                    print("if count not in candidates and count!=idx and count!=index: candidates.append(count)")
                                    candidates.append(count)
                                    if index not in candidates:
                                        candidates.append(index)#does basically the same, just marking it
                                else:
                                    print("ELSE: self.correct_in(idx)")
                                    self.correct_in(idx)
                        
                        count-=1
                        print("unterstes ende while: count", count)
                        print("."*40)
                    check-=1
                    print("+"*40)

                    if check<2 and len(candidates)<1:#clique begins with 3
                        self.remove_idx(idx)
                        print("Wenn check<2 und len(candidates)<1, dann remove_idx(idx) - normal, hier gibts nichts zu sehen")
                        return
                    print("+"*40)

                #print(candidates)
                #print("idx: ",idx)
                #print("index",index)
                for c in candidates:
                    if c in self.not_included:
                        candidates.remove(c)
        print("?"*40)
        
        candidates.sort()# because again, low indeces mean less connections, this is where we wlll start to eliminate
        print("Wir sind außerhalb der while. sortieren jetzt candidates um von rechts nach links zu arbeiten ",candidates,"\n\n")
        if idx not in candidates:
            candidates.append(idx)#we want it to be the most right one
        print("Wir appenden idx an candidates: ",candidates)
        checkbox = np.zeros((len(candidates),len(candidates)))
        
        for i in range(len(candidates)):
            for j in range(len(candidates)):
                checkbox[i][j]=self.sorted_numerated_out_matrix[candidates[i]][candidates[j]]!=0
        checkbox = np.array(checkbox,dtype=np.int32)
    
        print("jetzt haben wir ne Checkbox mit den richtigen EInträgen: \n",checkbox)
        
        real_candidates = np.array(checkbox.sum(axis=0)>1,dtype=np.int32)
        print("\n Und wir haben real_candidates = np.array(checkbox.sum(axis=0)>1,dtype=np.int32): ",real_candidates)
        not_suitable_candidates = []
        #for c in candidates[real_candidates]:
            #not_suitable_candidates.append(c)
        #candidates = candidates[real_candidates]
        #print(real_candidates)

        print("?"*40)
        print("&"*40)

        length = len(real_candidates)
        print("Die Länge davon: length = ",length)
        for i,candidate in enumerate(reversed(candidates)):
            c=length-i-1
            if real_candidates[c]:
                
                print(checkbox[:,c])
                if c == length-1:
                    continue
                print(len(checkbox[:,c]),checkbox[:,c])
                print(len(checkbox[:,length-1]),checkbox[:,length-1])
                check = checkbox[:,c]&checkbox[:,length-1]
                check_sum = (check).sum()
                print(check, check_sum)
                if check_sum < 1:
                    print("idx: ",candidates[i])
                    
        print("&"*40)

        #self.remove_idx(idx)
        self.update_nodes_of_interest()#at the end     
        
        
        
        
        
        
        
        
        
    def pick_last(self):
        last_candidate = len(self.outgoing_connections)-1
        self.check_idx(last_candidate)